In [ ]:
"""
@author: Zongyi Li and Daniel Zhengyu Huang
"""
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "4"

import torch
import torch.nn as nn
from typing import Union, List
import numpy as np
import time
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
class FactorizedSpectralConv3d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        modes: Union[int, List[int]],
        shifting_modes: int = 0,
        share_weights: bool = False,
    ):
        super().__init__()

        self.in_channels = int(in_channels)
        self.out_channels = int(out_channels)
        self.modes = (
            modes if isinstance(modes, (list, tuple)) else [modes, modes, modes]
        )
        self.shifting_modes = shifting_modes
        self.share_weights = share_weights

        if self.share_weights:
            self.weights = nn.Parameter(
                torch.randn(
                    self.in_channels,
                    self.out_channels,
                    self.modes[0],  # same for all
                    dtype=torch.cfloat,
                )
            )
        else:
            self.weights = nn.ParameterList(
                [
                    nn.Parameter(
                        torch.randn(
                            self.in_channels,
                            self.out_channels,
                            self.modes[i],
                            dtype=torch.cfloat,
                        )
                    )
                    for i in range(3)
                ]
            )

    def forward(self, x, upsample_factor):
        batch_size, _, d1, d2, d3 = x.shape
        dims = [d1, d2, d3]
        up_dims = [int(upsample_factor * d) for d in dims]

        axes = [-3, -2, -1]
        einsum_strs = ["bixyz,iox->boxyz", "bixyz,ioy->boxyz", "bixyz,ioz->boxyz"]
        results = []

        for i, axis in enumerate(axes):
            # Forward FFT along axis
            x_ft = torch.fft.rfft(x, dim=axis, norm="ortho")

            # Output shape in Fourier domain
            out_shape = [batch_size, self.out_channels, *up_dims]
            out_shape[axis] = up_dims[i] // 2 + 1
            out_ft = torch.zeros(
                *out_shape,
                dtype=torch.cfloat,
                device=x.device,
            )

            # Select correct weight
            weight = self.weights if self.share_weights else self.weights[i]

            # Fill Fourier output via einsum
            slices = [slice(None)] * 5
            slices[2 + i] = slice(
                self.shifting_modes, self.modes[i] + self.shifting_modes
            )

            out_ft[tuple(slices)] = torch.einsum(
                einsum_strs[i],
                x_ft[tuple(slices)],
                weight,
            )

            # Inverse FFT along axis
            x_rec = torch.fft.irfft(out_ft, n=up_dims[i], dim=axis, norm="ortho")
            results.append(x_rec)

        # Combine outputs from all axes
        return sum(results)


In [ ]:
class SpectralConv3d(nn.Module):
    """
    3D implementation of a spectral convolution,
    does a fft transform, convolves in that space and then brings it back,
    keep in mind the limitation with the number of modes,
    by ignoring high number modes we intent to get rid of the noise
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        modes1: int,
        modes2: int,
        modes3: int,
        shifting_modes: int,
    ):
        super(SpectralConv3d, self).__init__()

        # 3D Fourier layer. It does FFT, linear transform, and Inverse FFT.
        # in_channels: the number of input channels
        # out_channels: the number of output channels
        # modes1: the number of modes used for dimension 1, at most floor(N/2) + 1
        # modes2: the number of modes used for dimension 2, at most floor(N/2) + 1
        # shifting_modes: how many k to shift the modes from 0
        self.in_channels = int(in_channels)
        self.out_channels = int(out_channels)
        self.modes1 = modes1
        self.modes2 = modes2
        self.modes3 = modes3

        self.scale = (1 / (2 * in_channels)) ** (1.0 / 2.0)  # initialization scale
        self.weights1 = nn.Parameter(
            self.scale
            * (
                torch.randn(
                    self.in_channels,
                    self.out_channels,
                    self.modes1,
                    self.modes2,
                    self.modes3,
                    dtype=torch.cfloat,
                )
            )
        )
        self.weights2 = nn.Parameter(
            self.scale
            * (
                torch.randn(
                    self.in_channels,
                    self.out_channels,
                    self.modes1,
                    self.modes2,
                    self.modes3,
                    dtype=torch.cfloat,
                )
            )
        )
        self.weights3 = nn.Parameter(
            self.scale
            * (
                torch.randn(
                    self.in_channels,
                    self.out_channels,
                    self.modes1,
                    self.modes2,
                    self.modes3,
                    dtype=torch.cfloat,
                )
            )
        )
        self.weights4 = nn.Parameter(
            self.scale
            * (
                torch.randn(
                    self.in_channels,
                    self.out_channels,
                    self.modes1,
                    self.modes2,
                    self.modes3,
                    dtype=torch.cfloat,
                )
            )
        )
        self.shifting_modes = shifting_modes

    # Complex multiplication
    def compl_mul3d(self, input, weights):
        # (n_batch, in_channels, x, y), (in_channels, out_channels, x, y) -> (n_batch, out_channels, x, y)
        return torch.einsum("bixyz,ioxyz->boxyz", input, weights)

    def forward(self, x, upsample_factor):
        # x: input function (n_batch, in_channels, n_dim1, n_dim2)
        # upsample_factor: upsample factor for n_dim1 and n_dim2
        # dim1: scaled n_dim1 i.e. n_dim1 * upsample_factor
        # dim2: scaled n_dim2 i.e. n_dim2 * upsample_factor
        # dim3: scaled n_dim3
        # assert self.modes1 < dim1 // 2 and self.modes2 < dim2 // 2

        batch_size = x.shape[0]
        dim1 = int(upsample_factor * x.shape[2])
        dim2 = int(upsample_factor * x.shape[3])
        dim3 = int(upsample_factor * x.shape[4])

        # Compute Fourier coeffcients
        x_ft = torch.fft.rfftn(x, dim=[-3, -2, -1])
        # (n_batch, in_channels, n_dim1, n_dim2//2 + 1)

        shifting_modes = int(min(0, self.shifting_modes))
        modes1_use = int(min(self.modes1, dim1 // 2, x.shape[2] // 2)) + shifting_modes
        modes2_use = int(min(self.modes2, dim2 // 2, x.shape[3] // 2)) + shifting_modes
        modes3_use = (
            int(min(self.modes3, dim3 // 2 + 1, x.shape[4] // 2 + 1)) + shifting_modes
        )

        # Initialize output in frequency domain
        out_ft = torch.zeros(
            batch_size,
            self.out_channels,
            dim1,
            dim2,
            dim3 // 2 + 1,
            dtype=torch.cfloat,
            device=x.device,
        )

        # Region 1: low-low-low frequencies
        out_ft[
            :,
            :,
            shifting_modes:modes1_use,
            shifting_modes:modes2_use,
            shifting_modes:modes3_use,
        ] = self.compl_mul3d(
            x_ft[
                :,
                :,
                shifting_modes:modes1_use,
                shifting_modes:modes2_use,
                shifting_modes:modes3_use,
            ],
            self.weights1[
                :,
                :,
                shifting_modes:modes1_use,
                shifting_modes:modes2_use,
                shifting_modes:modes3_use,
            ],
        )

        # Region 2: high-low-low frequencies
        out_ft[
            :,
            :,
            -modes1_use:-shifting_modes,
            shifting_modes:modes2_use,
            shifting_modes:modes3_use,
        ] = self.compl_mul3d(
            x_ft[
                :,
                :,
                -modes1_use:-shifting_modes,
                shifting_modes:modes2_use,
                shifting_modes:modes3_use,
            ],
            self.weights2[
                :,
                :,
                -modes1_use:-shifting_modes,
                shifting_modes:modes2_use,
                shifting_modes:modes3_use,
            ],
        )

        # Region 3: low-high-low frequencies
        out_ft[
            :,
            :,
            shifting_modes:modes1_use,
            -modes2_use:-shifting_modes,
            shifting_modes:modes3_use,
        ] = self.compl_mul3d(
            x_ft[
                :,
                :,
                shifting_modes:modes1_use,
                -modes2_use:-shifting_modes,
                shifting_modes:modes3_use,
            ],
            self.weights3[
                :,
                :,
                shifting_modes:modes1_use,
                -modes2_use:-shifting_modes,
                shifting_modes:modes3_use,
            ],
        )

        # Region 4: high-high-low frequencies
        out_ft[
            :,
            :,
            -modes1_use:-shifting_modes,
            -modes2_use:-shifting_modes,
            shifting_modes:modes3_use,
        ] = self.compl_mul3d(
            x_ft[
                :,
                :,
                -modes1_use:-shifting_modes,
                -modes2_use:-shifting_modes,
                shifting_modes:modes3_use,
            ],
            self.weights4[
                :,
                :,
                -modes1_use:-shifting_modes,
                -modes2_use:-shifting_modes,
                shifting_modes:modes3_use,
            ],
        )

        # Return to physical space
        x = torch.fft.irfftn(out_ft, s=(dim1, dim2, dim3), dim=[-3, -2, -1]) * (
            upsample_factor**3
        )

        # Shape: (batch, out_channels, dim1, dim2, dim3)
        return x


In [ ]:
input_data = torch.rand(30, 3, 128, 128, 128).to(device)


In [ ]:
fspc = FactorizedSpectralConv3d(3, 3, 50, 10, False)
spc = SpectralConv3d(3, 3, 50, 50, 50, 10)

In [ ]:
fpsc_params = []
spc_params = []
modes_array = np.linspace(5, 100) 
for modes in modes_array:
    modes = int(modes)
    fspc = FactorizedSpectralConv3d(3, 3, modes, 10, False)
    spc = SpectralConv3d(3, 3, modes, modes, modes, 10)
    
    fpsc_params.append(sum(p.numel() for p in fspc.parameters()))
    spc_params.append(sum(p.numel() for p in spc.parameters()))


In [ ]:
fig, ax = plt.subplots()
ax.plot(modes_array, fpsc_params, label="factorized")
ax.plot(modes_array, spc_params, label="not factorized")
ax.set_yscale("log")
ax.legend()

In [ ]:
times_factorized = []
times_unfactorized = []
for _ in range(1000):
    start = time.time()
    with torch.no_grad():
        output = fspc(input_data, 1)
        times_factorized.append((time.time()-start)/100)
        del output
        start = time.time()
        output = spc(input_data, 1)
        times_unfactorized.append((time.time() - start) / 100)

print(sum(times_factorized)/100, sum(times_unfactorized)/100)

In [ ]:
del input_data
batch_size = 5
time_reps = 100
input_data = torch.rand(batch_size, 3, 212, 212, 212).to(device)
fpsc_params = []
spc_params = []
fpsc_times = []
spc_times = []
modes_array = np.linspace(5, 95, 15).astype(int)
with torch.no_grad():
    for modes in modes_array:
        print(modes, end = "\r")
        fspc = FactorizedSpectralConv3d(3, 3, modes, 10, False).to(device)
        spc = SpectralConv3d(3, 3, modes, modes, modes, 10).to(device)

        fpsc_params.append(sum(p.numel() for p in fspc.parameters()))
        spc_params.append(sum(p.numel() for p in spc.parameters()))
        for _ in range(time_reps):
            times_factorized = []
            times_unfactorized = []
            start = time.time()
            output = fspc(input_data, 1)
            times_factorized.append((time.time() - start) / batch_size)
            del output
            start = time.time()
            output = spc(input_data, 1)
            times_unfactorized.append((time.time() - start) / batch_size)  
            del output
        fpsc_times.append(sum(times_factorized)/time_reps)
        spc_times.append(sum(times_unfactorized) / time_reps)


In [ ]:
fig, ax = plt.subplots(1, 2)
ax[0].plot(modes_array, fpsc_params, label="factorized")
ax[0].plot(modes_array, spc_params, label="not factorized")
ax[0].set_yscale("log")
ax[0].legend()

ax[1].plot(modes_array, fpsc_times, label="factorized")
ax[1].plot(modes_array, spc_times, label="not factorized")
ax[1].set_yscale("log")
ax[1].legend()
